# Classificador de cladódios de pitaia

Notebook adaptado do fluxo de treinamento do projeto Dragon Fruit Application. O problema é classificação de imagens de cladódios (chamados de folhas nas pastas de dados), com MobileNetV3-Small pré-treinada, avaliação por grupos de origem e exportação ONNX.

A taxonomia tem nove doenças e uma classe Saudável. Brown Stem Spot é consolidada em Mancha-marrom; Gray Blight é consolidada em Queima dos ramos. Essas consolidações mantêm as dez saídas aprovadas e não afirmam equivalência etiológica entre as doenças.

O treino exige PyTorch compilado para ROCm e uma GPU disponível. O notebook interrompe a execução se ROCm não estiver ativo; não há fallback para CPU. Antes da primeira execução, use um ambiente Python com versões compatíveis de PyTorch ROCm e torchvision, além de Pillow, NumPy, scikit-learn, tqdm, onnx e onnxruntime. Os pesos ImageNet da MobileNetV3-Small podem ser baixados na primeira execução.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import math
import os
import random
import re
import unicodedata

import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.metrics import balanced_accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm

if getattr(torch.version, "hip", None) is None:
    raise RuntimeError(
        "Este treinamento exige uma compilação PyTorch ROCm. "
        "Instale PyTorch ROCm e torchvision compatíveis neste host."
    )
if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch ROCm não encontrou uma GPU utilizável. "
        "Confira /dev/kfd, /dev/dri e as permissões antes de treinar."
    )

DEVICE = torch.device("cuda:0")
torch.cuda.set_device(DEVICE)
torch.set_num_threads(2)
torch.set_num_interop_threads(1)
print(f"PyTorch: {torch.__version__} | ROCm/HIP: {torch.version.hip}")
print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")

CLASS_NAMES = [
    "Antracnose",
    "Cancro do caule",
    "Mancha branca",
    "Mancha-marrom",
    "Mancha preta",
    "Podridão da raiz",
    "Podridão do caule",
    "Podridão mole",
    "Queima dos ramos",
    "Saudável",
]
NUM_CLASSES = len(CLASS_NAMES)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
BATCH_SIZE = 32
MAX_EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 8
RESUME_EPOCHS = 15
RESUME_LEARNING_RATE = 2e-5
RESUME_PATIENCE = 6
IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

cwd_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_DIR = None
DATA_ROOT = None
for candidate in cwd_candidates:
    if PROJECT_DIR is None:
        if (candidate / "Dragon_Fruit_Full_Code.ipynb").is_file():
            PROJECT_DIR = candidate
        elif (candidate / "Dragon_Fruit_Application_And_Code" / "Dragon_Fruit_Full_Code.ipynb").is_file():
            PROJECT_DIR = candidate / "Dragon_Fruit_Application_And_Code"
    if DATA_ROOT is None and (candidate / "Folha").is_dir() and (
        candidate / "Dragon Fruit (Pitahaya)" / "Original Images"
    ).is_dir():
        DATA_ROOT = candidate

if PROJECT_DIR is None:
    raise FileNotFoundError("Não encontrei a pasta local Dragon_Fruit_Application_And_Code.")
if DATA_ROOT is None:
    raise FileNotFoundError(
        "Não encontrei Folha/ e Dragon Fruit (Pitahaya)/Original Images/. "
        "Abra este notebook dentro de /home/nicolas/Downloads/github."
    )

OUTPUT_DIR = PROJECT_DIR / "artifacts" / "pitaya"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "classificador_pitaia.pth"
ONNX_PATH = OUTPUT_DIR / "classificador_pitaia.onnx"
CLASS_MAP_PATH = OUTPUT_DIR / "classes_pt-BR.json"
METRICS_PATH = OUTPUT_DIR / "metricas.json"

CLASS_SOURCES = {
    "Antracnose": ["Anthracnose"],
    "Cancro do caule": ["Stem_Canker"],
    "Mancha branca": ["White Spot"],
    "Mancha-marrom": ["Brown Spot", "Brown Stem Spot"],
    "Mancha preta": ["Black Spot"],
    "Podridão da raiz": ["Root Rot"],
    "Podridão do caule": ["Stem Rot"],
    "Podridão mole": ["Soft Rot", "Soft_Rot"],
    "Queima dos ramos": ["Twig Blight", "Gray Blight"],
    "Saudável": ["Healthy"],
}
CLASS_MAP = {
    "model": "MobileNetV3-Small",
    "language": "pt-BR",
    "input": {
        "shape": [1, 3, 224, 224],
        "dtype": "float32",
        "color": "RGB",
        "resize": [224, 224],
        "normalization": {
            "mean": IMAGENET_MEAN,
            "std": IMAGENET_STD,
        },
    },
    "output": {
        "shape": [1, NUM_CLASSES],
        "dtype": "float32",
        "meaning": "logits; aplique softmax para obter probabilidades",
    },
    "classes": [
        {"index": index, "label": name, "source_labels": CLASS_SOURCES[name]}
        for index, name in enumerate(CLASS_NAMES)
    ],
    "taxonomy_note": (
        "Brown Stem Spot foi consolidada em Mancha-marrom e Gray Blight em "
        "Queima dos ramos para manter nove doenças e Saudável. "
        "O agrupamento de rótulos não confirma equivalência etiológica."
    ),
    "data_policy": {
        "included": ["Folha/", "Dragon Fruit (Pitahaya)/Original Images/"],
        "ignored": [
            "Dragon Fruit (Pitahaya)/Converted Images/",
            "Dragon Fruit (Pitahaya)/Annotated Files/",
            "arquivos Folha/*/aug_* pré-gerados",
        ],
        "split": "70/15/15 por grupo de origem, seed 42",
    },
}
with CLASS_MAP_PATH.open("w", encoding="utf-8") as f:
    json.dump(CLASS_MAP, f, ensure_ascii=False, indent=2)
print(f"Dados: {DATA_ROOT}")
print(f"Projeto: {PROJECT_DIR}")
print(f"Artefatos: {OUTPUT_DIR}")

## Preparação e deduplicação

São usadas as pastas Folha e Original Images. Converted Images e as anotações de caixas ficam de fora. Rotações com o mesmo nome-base são agrupadas, arquivos idênticos são detectados por SHA-256 e os nomes de arquivos compartilhados entre as duas fontes unem versões de resoluções diferentes.

Os arquivos aug_* prontos são excluídos; transformações aleatórias serão aplicadas apenas ao treino. Componentes com o mesmo conteúdo em classes diferentes são removidos por inteiro para não propagar rótulos contraditórios.

In [ ]:
FOLHA_TO_LABEL = {
    "anthracnose": "Antracnose",
    "cancro do caule": "Cancro do caule",
    "mancha branca": "Mancha branca",
    "mancha-marrom": "Mancha-marrom",
    "mancha_preta": "Mancha preta",
    "podridão da raiz": "Podridão da raiz",
    "podridão do caule": "Podridão do caule",
    "podridão-mole": "Podridão mole",
    "queima dos ramos": "Queima dos ramos",
    "saudável": "Saudável",
}
ORIGINAL_TO_LABEL = {
    "anthracnose": "Antracnose",
    "brown_stem_spot": "Mancha-marrom",
    "gray_blight": "Queima dos ramos",
    "healthy": "Saudável",
    "soft_rot": "Podridão mole",
    "stem_canker": "Cancro do caule",
}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
source_roots = [
    (DATA_ROOT / "Folha", FOLHA_TO_LABEL, "Folha"),
    (DATA_ROOT / "Dragon Fruit (Pitahaya)" / "Original Images", ORIGINAL_TO_LABEL, "Original Images"),
]
rows = []
skipped_augmented = 0
unreadable = []
for source_root, folder_map, source_name in source_roots:
    if not source_root.is_dir():
        raise FileNotFoundError(f"Pasta de imagens ausente: {source_root}")
    for class_dir in sorted(source_root.iterdir()):
        if not class_dir.is_dir():
            continue
        label = folder_map.get(class_dir.name.casefold())
        if label is None:
            continue
        for path in sorted(class_dir.rglob("*")):
            if not path.is_file() or path.suffix.casefold() not in IMAGE_EXTENSIONS:
                continue
            if source_name == "Folha" and path.stem.casefold().startswith("aug_"):
                skipped_augmented += 1
                continue
            base_stem = re.sub(
                r"_rot(?:0|90|180|270)$", "", path.stem, flags=re.IGNORECASE
            )
            normalized_stem = unicodedata.normalize("NFKC", base_stem).casefold()
            source_key = re.sub(r"[^a-z0-9]+", "_", normalized_stem).strip("_")
            try:
                with Image.open(path) as image:
                    width, height = image.size
                    image.verify()
                digest = hashlib.sha256()
                with path.open("rb") as image_file:
                    for chunk in iter(lambda: image_file.read(1024 * 1024), b""):
                        digest.update(chunk)
            except Exception as exc:
                unreadable.append((str(path), str(exc)))
                continue
            rows.append({
                "path": path,
                "relative_path": path.relative_to(DATA_ROOT).as_posix(),
                "label": label,
                "source_label": class_dir.name,
                "source": source_name,
                "source_key": source_key,
                "sha256": digest.hexdigest(),
                "pixels": width * height,
            })

if not rows:
    raise RuntimeError("Nenhuma imagem válida foi encontrada nas pastas configuradas.")

# Componentes conexos: mesmo rótulo + nome-base ou conteúdo idêntico em qualquer classe.
parent = list(range(len(rows)))
def find_root(index):
    if parent[index] != index:
        parent[index] = find_root(parent[index])
    return parent[index]

def union(left, right):
    left_root, right_root = find_root(left), find_root(right)
    if left_root != right_root:
        parent[right_root] = left_root

signatures = defaultdict(list)
for index, row in enumerate(rows):
    signatures[("name", row["label"], row["source_key"])].append(index)
    signatures[("sha256", row["sha256"])].append(index)
for indices in signatures.values():
    for index in indices[1:]:
        union(indices[0], index)

components = defaultdict(list)
for index, row in enumerate(rows):
    components[find_root(index)].append(row)

ambiguous_components = [
    component for component in components.values()
    if len({row["label"] for row in component}) > 1
]
usable_components = [
    component for component in components.values()
    if len({row["label"] for row in component}) == 1
]

def representative_key(row):
    is_original = row["source"] == "Original Images"
    is_unrotated = not re.search(r"_rot(?:0|90|180|270)$", row["path"].stem, re.IGNORECASE)
    return (
        not is_original,
        not is_unrotated,
        -row["pixels"],
        row["relative_path"],
    )

dataset_groups = []
for component in usable_components:
    label = component[0]["label"]
    representative = min(component, key=representative_key)
    group_material = "\n".join(sorted(row["relative_path"] for row in component))
    group_id = hashlib.sha256(group_material.encode("utf-8")).hexdigest()[:16]
    dataset_groups.append({
        "group_id": group_id,
        "path": representative["path"],
        "label": label,
        "label_idx": CLASS_NAMES.index(label),
        "source_paths": sorted(row["relative_path"] for row in component),
    })
dataset_groups.sort(key=lambda item: (item["label_idx"], item["group_id"]))

if unreadable:
    print(f"Aviso: {len(unreadable)} imagens não puderam ser lidas e foram ignoradas.")
print(f"Imagens válidas antes da deduplicação: {len(rows)}")
print(f"Grupos utilizáveis: {len(dataset_groups)}")
print(f"Arquivos aug_* ignorados: {skipped_augmented}")
print(
    f"Grupos com rótulos conflitantes removidos: {len(ambiguous_components)} "
    f"({sum(len(component) for component in ambiguous_components)} arquivos)"
)
for label in CLASS_NAMES:
    count = sum(group["label"] == label for group in dataset_groups)
    print(f"{label}: {count} grupos")

## Divisão por grupo de origem

A divisão é estratificada por classe. Todas as rotações, duplicatas exatas e cópias reconhecidas por nome-base ficam juntas no mesmo split. O arredondamento mantém proporções próximas a 70/15/15 e garante representação das classes nos três conjuntos.

In [ ]:
def split_sizes(count):
    train_count = math.floor(count * 0.70 + 0.5)
    val_count = math.floor(count * 0.15 + 0.5)
    test_count = count - train_count - val_count
    if min(train_count, val_count, test_count) < 1:
        raise ValueError(
            f"A classe tem somente {count} grupos; são necessários ao menos 3."
        )
    return train_count, val_count, test_count

groups_by_class = {
    label: [group for group in dataset_groups if group["label"] == label]
    for label in CLASS_NAMES
}
train_groups, val_groups, test_groups = [], [], []
for class_index, label in enumerate(CLASS_NAMES):
    class_groups = sorted(groups_by_class[label], key=lambda item: item["group_id"])
    if len(class_groups) < 3:
        raise ValueError(f"{label} tem menos de três grupos independentes.")
    random.Random(SEED + class_index).shuffle(class_groups)
    train_count, val_count, test_count = split_sizes(len(class_groups))
    train_groups.extend(class_groups[:train_count])
    val_groups.extend(class_groups[train_count:train_count + val_count])
    test_groups.extend(class_groups[train_count + val_count:])

train_group_ids = {group["group_id"] for group in train_groups}
val_group_ids = {group["group_id"] for group in val_groups}
test_group_ids = {group["group_id"] for group in test_groups}
assert not train_group_ids & val_group_ids
assert not train_group_ids & test_group_ids
assert not val_group_ids & test_group_ids
assert len(train_groups) + len(val_groups) + len(test_groups) == len(dataset_groups)

print(
    f"Treino: {len(train_groups)} grupos ({len(train_groups)/len(dataset_groups):.1%}) | "
    f"validação: {len(val_groups)} ({len(val_groups)/len(dataset_groups):.1%}) | "
    f"teste: {len(test_groups)} ({len(test_groups)/len(dataset_groups):.1%})"
)
print(f"{'Classe':<22} {'Treino':>8} {'Validação':>11} {'Teste':>8}")
for label in CLASS_NAMES:
    counts = [
        sum(group["label"] == label for group in split)
        for split in (train_groups, val_groups, test_groups)
    ]
    print(f"{label:<22} {counts[0]:>8} {counts[1]:>11} {counts[2]:>8}")

## Conjunto residente na GPU e transformações CUDA/ROCm

Cada imagem é decodificada uma vez na CPU, transferida imediatamente e redimensionada na GPU. O conjunto final em uint8 permanece na VRAM durante treino e avaliação; não há DataLoader por época nem workers de CPU. Os aumentos aleatórios e a normalização são vetorizados na GPU.


In [ ]:
import torch.nn.functional as F

GPU_MEAN = torch.tensor(IMAGENET_MEAN, dtype=torch.float32, device=DEVICE).view(1, 3, 1, 1)
GPU_STD = torch.tensor(IMAGENET_STD, dtype=torch.float32, device=DEVICE).view(1, 3, 1, 1)
PIL_TO_TENSOR = transforms.PILToTensor()

@torch.inference_mode()
def preload_groups(groups, image_size, split_name):
    images_gpu = torch.empty(
        (len(groups), 3, image_size, image_size), dtype=torch.uint8, device=DEVICE
    )
    labels_gpu = torch.tensor(
        [group["label_idx"] for group in groups], dtype=torch.long, device=DEVICE
    )
    for index, group in enumerate(tqdm(groups, desc=f"Carregando {split_name} na VRAM")):
        with Image.open(group["path"]) as image:
            decoded_cpu = PIL_TO_TENSOR(image.convert("RGB"))
        decoded_gpu = decoded_cpu.unsqueeze(0).to(DEVICE, dtype=torch.float32)
        resized_gpu = F.interpolate(
            decoded_gpu,
            size=(image_size, image_size),
            mode="bilinear",
            align_corners=False,
            antialias=True,
        )
        images_gpu[index].copy_(resized_gpu[0].round().clamp_(0, 255).to(torch.uint8))
        del decoded_cpu, decoded_gpu, resized_gpu
    assert images_gpu.device == DEVICE and labels_gpu.device == DEVICE
    return {"images": images_gpu, "labels": labels_gpu}

train_gpu = preload_groups(train_groups, 256, "treino")
val_gpu = preload_groups(val_groups, IMAGE_SIZE, "validação")
test_gpu = preload_groups(test_groups, IMAGE_SIZE, "teste")
torch.cuda.synchronize(DEVICE)

train_counts = np.bincount(
    [group["label_idx"] for group in train_groups], minlength=NUM_CLASSES
)
class_weights = torch.tensor(
    len(train_groups) / (NUM_CLASSES * train_counts),
    dtype=torch.float32,
    device=DEVICE,
)

resident_mib = (
    train_gpu["images"].numel()
    + val_gpu["images"].numel()
    + test_gpu["images"].numel()
) / (1024 ** 2)
print(f"Imagens e rótulos residentes em {DEVICE}: {resident_mib:.1f} MiB de pixels uint8")
print(f"Memória máxima alocada na GPU durante a carga: {torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 3):.2f} GiB")
print(f"Pesos de classe na GPU: {class_weights.detach().cpu().numpy().round(3)}")


def _adjust_hue_per_image(images, hue_factor):
    red, green, blue = images.unbind(dim=1)
    maximum, max_index = images.max(dim=1)
    minimum = images.min(dim=1).values
    delta = maximum - minimum
    safe_delta = delta.clamp_min(1e-8)

    hue_r = torch.remainder((green - blue) / safe_delta, 6.0)
    hue_g = (blue - red) / safe_delta + 2.0
    hue_b = (red - green) / safe_delta + 4.0
    hue = torch.where(max_index == 0, hue_r, torch.where(max_index == 1, hue_g, hue_b))
    hue = torch.where(delta > 0, hue / 6.0, torch.zeros_like(hue))
    hue = torch.remainder(hue + hue_factor[:, None, None], 1.0)

    saturation = delta / maximum.clamp_min(1e-8)
    chroma = maximum * saturation
    hue_sector = hue * 6.0
    intermediate = chroma * (1.0 - torch.abs(torch.remainder(hue_sector, 2.0) - 1.0))
    sector = torch.floor(hue_sector).to(torch.int64).remainder(6)
    zero = torch.zeros_like(chroma)
    out_r = torch.where((sector == 0) | (sector == 5), chroma,
             torch.where((sector == 1) | (sector == 4), intermediate, zero))
    out_g = torch.where((sector == 1) | (sector == 2), chroma,
             torch.where((sector == 0) | (sector == 3), intermediate, zero))
    out_b = torch.where((sector == 3) | (sector == 4), chroma,
             torch.where((sector == 2) | (sector == 5), intermediate, zero))
    match = (maximum - chroma).unsqueeze(1)
    return torch.stack((out_r, out_g, out_b), dim=1) + match


def augment_batch_gpu(images_uint8):
    assert images_uint8.device == DEVICE and images_uint8.dtype == torch.uint8
    batch_size, _, height, width = images_uint8.shape
    output_size = IMAGE_SIZE
    images = images_uint8.to(torch.float32).div_(255.0)

    # RandomResizedCrop com parâmetros independentes para cada imagem.
    area_scale = 0.80 + 0.20 * torch.rand(batch_size, device=DEVICE)
    log_ratio_low, log_ratio_high = math.log(0.90), math.log(1.10)
    ratio = torch.exp(
        log_ratio_low + (log_ratio_high - log_ratio_low) * torch.rand(batch_size, device=DEVICE)
    )
    crop_w = torch.sqrt(area_scale * ratio).mul(width).round().to(torch.long).clamp_(1, width)
    crop_h = torch.sqrt(area_scale / ratio).mul(height).round().to(torch.long).clamp_(1, height)
    left = (torch.rand(batch_size, device=DEVICE) * (width - crop_w + 1)).to(torch.long)
    top = (torch.rand(batch_size, device=DEVICE) * (height - crop_h + 1)).to(torch.long)

    coordinates = (2.0 * (torch.arange(output_size, device=DEVICE, dtype=torch.float32) + 0.5) / output_size) - 1.0
    center_x = 2.0 * (left.to(torch.float32) + crop_w.to(torch.float32) / 2.0) / width - 1.0
    center_y = 2.0 * (top.to(torch.float32) + crop_h.to(torch.float32) / 2.0) / height - 1.0
    grid_x = center_x[:, None, None] + (crop_w.to(torch.float32) / width)[:, None, None] * coordinates[None, None, :]
    grid_y = center_y[:, None, None] + (crop_h.to(torch.float32) / height)[:, None, None] * coordinates[None, :, None]
    grid = torch.stack((grid_x.expand(-1, output_size, -1), grid_y.expand(-1, -1, output_size)), dim=-1)
    images = F.grid_sample(images, grid, mode="bilinear", padding_mode="zeros", align_corners=False)

    flip_h = torch.rand(batch_size, device=DEVICE) < 0.5
    flip_v = torch.rand(batch_size, device=DEVICE) < 0.2
    images = torch.where(flip_h[:, None, None, None], images.flip(-1), images)
    images = torch.where(flip_v[:, None, None, None], images.flip(-2), images)

    angles = (torch.rand(batch_size, device=DEVICE) * 24.0 - 12.0) * (math.pi / 180.0)
    affine = torch.zeros((batch_size, 2, 3), dtype=torch.float32, device=DEVICE)
    affine[:, 0, 0] = torch.cos(angles)
    affine[:, 0, 1] = -torch.sin(angles)
    affine[:, 1, 0] = torch.sin(angles)
    affine[:, 1, 1] = torch.cos(angles)
    rotation_grid = F.affine_grid(affine, images.shape, align_corners=False)
    images = F.grid_sample(images, rotation_grid, mode="bilinear", padding_mode="zeros", align_corners=False)

    brightness = 0.85 + 0.30 * torch.rand(batch_size, device=DEVICE)
    images = (images * brightness[:, None, None, None]).clamp_(0.0, 1.0)
    contrast = 0.85 + 0.30 * torch.rand(batch_size, device=DEVICE)
    luminance = (0.2989 * images[:, 0] + 0.5870 * images[:, 1] + 0.1140 * images[:, 2]).mean(dim=(1, 2), keepdim=True)
    images = ((images - luminance[:, None]) * contrast[:, None, None, None] + luminance[:, None]).clamp_(0.0, 1.0)
    saturation = 0.88 + 0.24 * torch.rand(batch_size, device=DEVICE)
    gray = (0.2989 * images[:, 0] + 0.5870 * images[:, 1] + 0.1140 * images[:, 2]).unsqueeze(1)
    images = ((images - gray) * saturation[:, None, None, None] + gray).clamp_(0.0, 1.0)
    hue = (torch.rand(batch_size, device=DEVICE) * 0.03) - 0.015
    images = _adjust_hue_per_image(images, hue).clamp_(0.0, 1.0)
    return (images - GPU_MEAN) / GPU_STD


def normalize_batch_gpu(images_uint8):
    assert images_uint8.device == DEVICE and images_uint8.dtype == torch.uint8
    images = images_uint8.to(torch.float32).div_(255.0)
    return (images - GPU_MEAN) / GPU_STD


## Modelo pré-treinado

A cabeça classificadora da MobileNetV3-Small é substituída por uma camada de dez logits. O restante da rede parte dos pesos ImageNet e é ajustado com os cladódios.

In [ ]:
weights = MobileNet_V3_Small_Weights.DEFAULT
model = mobilenet_v3_small(weights=weights)
model.classifier[-1] = torch.nn.Linear(
    model.classifier[-1].in_features, NUM_CLASSES
)
model = model.to(DEVICE)
RESUMING_CHECKPOINT = CHECKPOINT_PATH.is_file()
if RESUMING_CHECKPOINT:
    model.load_state_dict(
        torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
    )
    print("Retomando do melhor checkpoint existente na GPU ROCm.")

trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f"Parâmetros treináveis: {trainable_params:,}")
print(f"Saída configurada: {NUM_CLASSES} logits")
print(f"Checkpoint: {CHECKPOINT_PATH}")

## Treino ROCm com checkpoint recuperável

O melhor checkpoint é carregado na GPU. A seleção do lote usa um microbenchmark apenas ROCm; se faltar VRAM, tenta um lote menor na própria GPU. A parada antecipada mantém o melhor estado de validação.


In [ ]:
import time

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
training_epochs = RESUME_EPOCHS if RESUMING_CHECKPOINT else MAX_EPOCHS
training_patience = RESUME_PATIENCE if RESUMING_CHECKPOINT else PATIENCE
training_lr = RESUME_LEARNING_RATE if RESUMING_CHECKPOINT else LEARNING_RATE
initial_state = {key: value.detach().clone() for key, value in model.state_dict().items()}


def calibrate_batch_size(candidates=(256, 128, 64, 32)):
    total_vram = torch.cuda.get_device_properties(DEVICE).total_memory
    results = []
    for candidate in candidates:
        model.load_state_dict(initial_state)
        optimizer_probe = torch.optim.AdamW(
            model.parameters(), lr=training_lr, weight_decay=WEIGHT_DECAY
        )
        sample_count = min(candidate, len(train_gpu["labels"]))
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(DEVICE)
            model.train()
            for _ in range(2):
                indices = torch.randperm(len(train_gpu["labels"]), device=DEVICE)[:sample_count]
                batch_images = augment_batch_gpu(train_gpu["images"].index_select(0, indices))
                batch_labels = train_gpu["labels"].index_select(0, indices)
                optimizer_probe.zero_grad(set_to_none=True)
                loss = criterion(model(batch_images), batch_labels)
                loss.backward()
                optimizer_probe.step()
            torch.cuda.synchronize(DEVICE)
            peak_bytes = torch.cuda.max_memory_allocated(DEVICE)
            if peak_bytes > total_vram * 0.70:
                print(f"Lote {candidate}: descartado, pico de VRAM {peak_bytes / total_vram:.1%}.")
                del optimizer_probe, batch_images, batch_labels, loss
                torch.cuda.empty_cache()
                continue

            start = time.perf_counter()
            for _ in range(3):
                indices = torch.randperm(len(train_gpu["labels"]), device=DEVICE)[:sample_count]
                batch_images = augment_batch_gpu(train_gpu["images"].index_select(0, indices))
                batch_labels = train_gpu["labels"].index_select(0, indices)
                optimizer_probe.zero_grad(set_to_none=True)
                loss = criterion(model(batch_images), batch_labels)
                loss.backward()
                optimizer_probe.step()
            torch.cuda.synchronize(DEVICE)
            elapsed = time.perf_counter() - start
            peak_bytes = max(peak_bytes, torch.cuda.max_memory_allocated(DEVICE))
            rate = (sample_count * 3) / elapsed
            results.append((rate, candidate, peak_bytes))
            print(
                f"Lote {candidate}: {rate:.1f} imagens/s, "
                f"pico alocado {peak_bytes / (1024 ** 3):.2f} GiB."
            )
            del optimizer_probe, batch_images, batch_labels, loss
            torch.cuda.empty_cache()
        except RuntimeError as error:
            del optimizer_probe
            torch.cuda.empty_cache()
            model.load_state_dict(initial_state)
            if "out of memory" not in str(error).casefold():
                raise
            print(f"Lote {candidate}: sem VRAM suficiente; tentando um lote menor na GPU.")
    if not results:
        raise RuntimeError("Nenhum tamanho de lote coube na GPU ROCm; treino não será movido para CPU.")
    _, chosen_batch, _ = max(results)
    model.load_state_dict(initial_state)
    model.eval()
    return chosen_batch


BATCH_SIZE = calibrate_batch_size()
print(f"Lote ROCm escolhido: {BATCH_SIZE}")
assert DEVICE.type == "cuda" and getattr(torch.version, "hip", None) is not None
assert next(model.parameters()).device == DEVICE

optimizer = torch.optim.AdamW(
    model.parameters(), lr=training_lr, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=training_epochs
)


class EarlyStopping:
    def __init__(self, patience, path):
        self.patience = patience
        self.path = Path(path)
        self.best_loss = float("inf")
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            state = {key: value.detach().cpu() for key, value in model.state_dict().items()}
            torch.save(state, self.path)
            print(f"Melhor checkpoint salvo: val_loss={val_loss:.5f}")
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


def run_epoch(dataset, training):
    model.train(training)
    images_all, labels_all = dataset["images"], dataset["labels"]
    count = len(labels_all)
    order = torch.randperm(count, device=DEVICE) if training else torch.arange(count, device=DEVICE)
    loss_sum = torch.zeros((), dtype=torch.float32, device=DEVICE)
    correct_sum = torch.zeros((), dtype=torch.long, device=DEVICE)
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for start in tqdm(range(0, count, BATCH_SIZE), leave=False):
            indices = order[start:start + BATCH_SIZE]
            raw_images = images_all.index_select(0, indices)
            batch_labels = labels_all.index_select(0, indices)
            batch_images = augment_batch_gpu(raw_images) if training else normalize_batch_gpu(raw_images)
            assert batch_images.device == DEVICE and batch_labels.device == DEVICE
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(batch_images)
            loss = criterion(logits, batch_labels)
            if training:
                loss.backward()
                optimizer.step()
            batch_count = batch_labels.shape[0]
            loss_sum.add_(loss.detach() * batch_count)
            correct_sum.add_((logits.argmax(dim=1) == batch_labels).sum())
    totals = torch.stack((loss_sum, correct_sum.to(torch.float32))).detach().cpu().tolist()
    return totals[0] / count, totals[1] / count


if RESUMING_CHECKPOINT:
    baseline_val_loss, baseline_val_accuracy = run_epoch(val_gpu, training=False)
    early_stopping = EarlyStopping(training_patience, CHECKPOINT_PATH)
    early_stopping.best_loss = baseline_val_loss
    print(
        f"Checkpoint base: val_loss={baseline_val_loss:.5f}, "
        f"val_acc={baseline_val_accuracy:.3f}; "
        f"continuando por até {training_epochs} épocas na GPU ROCm."
    )
else:
    early_stopping = EarlyStopping(training_patience, CHECKPOINT_PATH)

history = {"train_loss": [], "train_accuracy": [], "val_loss": [], "val_accuracy": []}
for epoch in range(1, training_epochs + 1):
    train_loss, train_accuracy = run_epoch(train_gpu, training=True)
    val_loss, val_accuracy = run_epoch(val_gpu, training=False)
    scheduler.step()
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)
    print(
        f"Época {epoch:02d}/{training_epochs} | "
        f"treino loss={train_loss:.4f}, acc={train_accuracy:.3f} | "
        f"val loss={val_loss:.4f}, acc={val_accuracy:.3f}"
    )
    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print(f"Parada antecipada na época {epoch}.")
        break

if not CHECKPOINT_PATH.is_file():
    raise RuntimeError("O treinamento terminou sem produzir o checkpoint.")
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True))
model.eval()
assert next(model.parameters()).device == DEVICE
print(f"Treino ROCm concluído. Melhor checkpoint: {CHECKPOINT_PATH}")


## Métricas por classe

Macro F1 e acurácia balanceada dão o mesmo peso às classes, além da acurácia geral. A tabela também registra precisão, recall, F1 e suporte por rótulo.

In [ ]:
@torch.inference_mode()
def detailed_evaluation(model, dataset, split_name):
    model.eval()
    images_all, labels_all = dataset["images"], dataset["labels"]
    predictions = []
    for start in tqdm(range(0, len(labels_all), BATCH_SIZE), desc=f"Avaliando {split_name}"):
        batch_images = normalize_batch_gpu(images_all[start:start + BATCH_SIZE])
        logits = model(batch_images)
        predictions.append(logits.argmax(dim=1))
    y_pred_gpu = torch.cat(predictions)
    y_true = labels_all.detach().cpu().numpy().tolist()
    y_pred = y_pred_gpu.detach().cpu().numpy().tolist()

    label_ids = list(range(NUM_CLASSES))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=label_ids, zero_division=0
    )
    accuracy = float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))
    balanced_accuracy = float(balanced_accuracy_score(y_true, y_pred))
    macro_f1 = float(np.mean(f1))
    per_class = []
    for index, name in enumerate(CLASS_NAMES):
        per_class.append({
            "label": name,
            "precision": float(precision[index]),
            "recall": float(recall[index]),
            "f1": float(f1[index]),
            "support": int(support[index]),
        })
    result = {
        "split": split_name,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "macro_f1": macro_f1,
        "per_class": per_class,
        "y_true": y_true,
        "y_pred": y_pred,
    }
    print(
        f"{split_name}: accuracy={accuracy:.4f} | "
        f"balanced_accuracy={balanced_accuracy:.4f} | macro_F1={macro_f1:.4f}"
    )
    print(f"{'Classe':<22} {'Precisão':>10} {'Recall':>10} {'F1':>10} {'Suporte':>9}")
    for row in per_class:
        print(
            f"{row['label']:<22} {row['precision']:>10.4f} "
            f"{row['recall']:>10.4f} {row['f1']:>10.4f} {row['support']:>9}"
        )
    return result

validation_results = detailed_evaluation(model, val_gpu, "validação")
test_results = detailed_evaluation(model, test_gpu, "teste")
metrics_report = {
    "model": "MobileNetV3-Small",
    "checkpoint": str(CHECKPOINT_PATH),
    "groups": {
        "train": len(train_groups),
        "validation": len(val_groups),
        "test": len(test_groups),
    },
    "validation": {
        key: value for key, value in validation_results.items()
        if key not in {"y_true", "y_pred"}
    },
    "test": {
        key: value for key, value in test_results.items()
        if key not in {"y_true", "y_pred"}
    },
}
with METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(metrics_report, f, ensure_ascii=False, indent=2)
print(f"Métricas salvas em: {METRICS_PATH}")


## Exportação ONNX e equivalência numérica

O modelo exportado tem entrada estática float32 [1, 3, 224, 224] e saída [1, 10]. A validação executa a mesma imagem no checkpoint PyTorch e no ONNX Runtime CPU e verifica a proximidade dos logits.

In [ ]:
import onnx
import onnxruntime as ort

model.eval()
dummy_input = torch.zeros((1, 3, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.float32, device=DEVICE)
assert dummy_input.device == DEVICE and next(model.parameters()).device == DEVICE
torch.onnx.export(
    model,
    dummy_input,
    str(ONNX_PATH),
    input_names=["imagem"],
    output_names=["logits"],
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
)
onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])

sample_tensor = normalize_batch_gpu(val_gpu["images"][:1])
with torch.inference_mode():
    pytorch_logits = model(sample_tensor).cpu().numpy()
sample_numpy = sample_tensor.detach().cpu().numpy().astype(np.float32, copy=False)
onnx_logits = session.run(["logits"], {"imagem": sample_numpy})[0]
assert pytorch_logits.shape == (1, NUM_CLASSES), pytorch_logits.shape
assert onnx_logits.shape == (1, NUM_CLASSES), onnx_logits.shape
max_abs_difference = float(np.max(np.abs(pytorch_logits - onnx_logits)))
np.testing.assert_allclose(pytorch_logits, onnx_logits, rtol=1e-3, atol=1e-4)
print(f"ONNX validado: {ONNX_PATH}")
print(f"Entrada: {session.get_inputs()[0].shape}, saída: {onnx_logits.shape}")
print(f"Maior diferença absoluta PyTorch ROCm/ONNX: {max_abs_difference:.8g}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Mapa pt-BR: {CLASS_MAP_PATH}")


## Artefatos gerados

A pasta artifacts/pitaya contém o checkpoint PyTorch, o modelo ONNX, o mapa pt-BR dos dez logits e as métricas de validação/teste. O notebook exige PyTorch ROCm; as imagens ficam na VRAM entre as épocas e o treino falha claramente se a GPU não estiver disponível.
